# Replay a kernel's history

A saved notebook shows, beside each cell, a count and an output, and both come from the
history of runs, not from the code as it reads now. Here you replay a history of runs the way
a kernel would, and then find the cells whose shown output a restart and run all would not
give back.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import contextlib
import io

def run_cell(source, namespace):
    """Run one cell's code in namespace and return (shown, failed).

    shown is the text the cell printed, stripped, and failed is False; or, when the code
    raises, shown is the exception's class name, such as "NameError", and failed is True.
    """
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            exec(source, namespace)
    except Exception as error:
        return type(error).__name__, True
    return printed.getvalue().strip(), False

# Five cells, top to bottom, and the order a student ran them in, starting from a fresh kernel.
sources = [
    "total = 0",                              # cell 0
    "step = 4",                               # cell 1
    "total = total + step\nprint(total)",     # cell 2
    "print(total * scale)",                   # cell 3
    "scale = 10",                             # cell 4
]
history = [0, 1, 2, 3, 4, 2, 3]
print(run_cell("print(6 * 7)", {}), run_cell("print(undefined_name)", {}))

## Part 1 (coding task): What the notebook shows

Write `replay(sources, history)`. `history` lists the cells in the order they were run,
starting from a fresh kernel, and a cell can appear in it more than once. Run each one with
`run_cell`. A run that raises still takes an execution count.

Return `(counts, outputs)`, two lists with one entry per cell: `counts[i]` is the execution
count shown beside cell `i`, and `outputs[i]` is the text shown under it. Both are `None`
for a cell that never ran.

In [ ]:
def replay(sources, history):
    # TODO
    raise NotImplementedError

In [ ]:
counts, outputs = replay(sources, history)
print("counts: ", counts)
print("outputs:", outputs)
assert len(counts) == len(sources) and len(outputs) == len(sources), "return one count and one output per cell"
assert outputs[2] != "NameError", "cell 2 cannot find total: every run in the history uses the same kernel, so the names earlier runs assigned are still there"
assert counts != [1, 1, 2, 2, 1], f"counts {counts} say how often each cell ran: an execution count goes up by one with every run in the kernel, whichever cell it is"
assert counts[0] == 1, f"counts {counts}: the first run takes count 1"
assert counts != [1, 2, 5, 6, 4], f"counts {counts}: the run of cell 3 that raised takes a count too"
assert counts == [1, 2, 6, 7, 5], f"counts {counts}, expected [1, 2, 6, 7, 5]"
assert outputs[2] == "8", f"cell 2 shows {outputs[2]!r}: show only what its latest run printed"
assert outputs[3] == "80", f"cell 3 shows {outputs[3]!r}: its latest run came after scale was assigned"
assert outputs == ["", "", "8", "80", ""], f"outputs {outputs}, expected ['', '', '8', '80', '']"
counts, outputs = replay(["a = 1", "print(a + 1)", "b = 2"], [0, 1])
assert counts == [1, 2, None] and outputs == ["", "2", None], "a cell that never ran has count None and output None"
print("Part 1 passed")

## Part 2 (coding task): What a fresh run would not reproduce

Restart session and run all starts a new kernel and runs the cells from the top. It stops at
the first cell that raises, and the cells below that one do not run at all.

Write `unreproducible(sources, history)`. Return, in increasing order, the cells that ran in
`history` whose shown output a restart and run all would not show again: the fresh run either
shows something different under the cell or never reaches it.

In [ ]:
def unreproducible(sources, history):
    # TODO
    raise NotImplementedError

In [ ]:
r = unreproducible(sources, history)
print("cells a fresh run would not reproduce:", r)
assert isinstance(r, list), f"return a list of cell indices, got {r!r}"
assert 2 in r, "cell 2 shows 8, but a fresh run shows 4"
assert 3 in r, "cell 3 shows 80, but a fresh run stops there with NameError"
r2 = unreproducible(["print(x)", "x = 5"], [1, 0])
assert r2 == [0, 1], f"a notebook whose cell 1 ran before cell 0 gives {r2}, expected [0, 1]: a restart starts from an empty namespace, so cell 0 raises NameError and the run stops before cell 1"
assert 4 in r, "the fresh run stops at cell 3's NameError and never reaches cell 4: stop at the first cell that raises"
assert r == [2, 3, 4], f"expected [2, 3, 4], got {r}"
assert unreproducible(["print(nothing_yet)", "nothing_yet = 1"], [0, 1]) == [1], "a cell that raises the same error both in the history and after the restart is reproduced: only the cells the fresh run never reaches differ"
r = unreproducible(["x = 2", "print(x)", "print(x + 1)"], [0, 1])
assert r == [], f"got {r}: cell 2 never ran in the history, so it shows nothing to compare"
r = unreproducible(["x = 2", "print(x)"], [0, 1, 0, 1])
assert r == [], f"got {r}: the counts change after any restart, so compare what the cells show"
print("Part 2 passed")

## Solution

### Part 1: What the notebook shows

In [ ]:
def replay(sources, history):
    namespace = {}                   # one kernel for the whole history
    counts = [None] * len(sources)
    outputs = [None] * len(sources)
    for count, i in enumerate(history, start=1):
        outputs[i], _ = run_cell(sources[i], namespace)
        counts[i] = count
    return counts, outputs

**Why it works.** One `namespace` dictionary lives for the whole history, as one kernel does, so cell 2
finds the `total` and `step` that earlier runs assigned, and its second run adds 4 to the 4
the first run left. `enumerate(history, start=1)` gives every run the next count, including
cell 3's first run, which raised. Assigning `counts[i]` and `outputs[i]` on every run keeps
only the latest, which is what the notebook shows: cell 2 shows 8 with count 6, and cell 3
shows 80 with count 7, though in the notebook's top to bottom order it reads a `scale` that
only cell 4, below it, assigns.

### Part 2: What a fresh run would not reproduce

In [ ]:
def unreproducible(sources, history):
    counts, shown = replay(sources, history)
    namespace = {}                   # the restarted kernel
    fresh = [None] * len(sources)    # None: the fresh run never gets there
    for i, source in enumerate(sources):
        fresh[i], failed = run_cell(source, namespace)
        if failed:
            break
    return [i for i in range(len(sources)) if counts[i] is not None and shown[i] != fresh[i]]

**Why it works.** `replay` gives what the notebook shows now. The fresh run uses a new dictionary, the
restarted kernel, and the `break` on the first failure leaves `None` for every cell below it,
so a cell the fresh run never reaches can never match what it showed. Only cells with a count
take part, because a cell that never ran shows nothing to compare. In the student's notebook,
cell 2's 8 comes from running it twice, cell 3's 80 from a `scale` assigned below it, and
cell 4 sits under the cell where a fresh run stops. A grader running the notebook would see
4 and then a `NameError`.